# Blood Cell Classification - Exploratory Data Analysis

This notebook explores the dataset before training.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

# Add src to path
sys.path.append('../src')

from config import CONFIG

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Set paths
data_dir = '../data'
splits = ['train', 'val', 'test']
classes = ['eosinophil', 'lymphocyte', 'monocyte', 'neutrophil']

print(f"Data directory: {os.path.abspath(data_dir)}")
for split in splits:
    split_path = os.path.join(data_dir, split)
    if os.path.exists(split_path):
        print(f"{split}: {len(os.listdir(split_path))} classes")
    else:
        print(f"{split}: NOT FOUND")

In [ ]:
# 1. Class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, split in enumerate(splits):
    counts = []
    for class_name in classes:
        class_dir = os.path.join(data_dir, split, class_name)
        if os.path.exists(class_dir):
            counts.append(len(os.listdir(class_dir)))
        else:
            counts.append(0)
    
    axes[idx].bar(classes, counts, color=['#4e79a7', '#f28e2b', '#e15759', '#76b7b2'])
    axes[idx].set_title(f'{split.upper()} Set', fontweight='bold', fontsize=12)
    axes[idx].set_ylabel('Number of images')
    axes[idx].tick_params(axis='x', rotation=45)
    
    for i, v in enumerate(counts):
        axes[idx].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.suptitle('Dataset Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Sample images from each class
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for row, class_name in enumerate(classes):
    class_dir = os.path.join(data_dir, 'train', class_name)
    if not os.path.exists(class_dir):
        continue
    
    images = os.listdir(class_dir)[:4]
    
    for col, img_name in enumerate(images):
        img_path = os.path.join(class_dir, img_name)
        img = Image.open(img_path).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{class_name}\n{img.size}', fontsize=9)
        axes[row, col].axis('off')

plt.suptitle('Sample Training Images by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Image size distribution
print("Image size analysis:")
print("-" * 40)

size_counts = {}

for class_name in classes:
    class_dir = os.path.join(data_dir, 'train', class_name)
    if not os.path.exists(class_dir):
        continue
    
    for img_name in os.listdir(class_dir)[:50]:
        img_path = os.path.join(class_dir, img_name)
        img = Image.open(img_path)
        size = img.size  # (width, height)
        size_counts[size] = size_counts.get(size, 0) + 1

print(f"Unique image sizes: {list(size_counts.keys())}")
print(f"Most common size: {max(size_counts, key=size_counts.get)}")

In [ ]:
# 4. Visualise augmentation pipeline
from dataset import get_transforms
from config import MEAN, STD

aug_transform = get_transforms(224, mode='train')
val_transform = get_transforms(224, mode='val')

# Load one image
sample_img_path = os.path.join(data_dir, 'train', 'neutrophil', 
                               os.listdir(os.path.join(data_dir, 'train', 'neutrophil'))[0])
sample_img = Image.open(sample_img_path).convert('RGB')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax in axes.flat:
    aug_img = aug_transform(sample_img)
    # Denormalise
    arr = aug_img.numpy().transpose(1, 2, 0)
    mean = np.array(MEAN)
    std = np.array(STD)
    arr = np.clip(arr * std + mean, 0, 1)
    ax.imshow(arr)
    ax.axis('off')

plt.suptitle('8 Different Augmentations of the Same Image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../augmentation_samples.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Compute dataset statistics (run once, then update config)
from dataset import compute_dataset_stats

# Uncomment to compute your dataset's mean and std
# mean, std = compute_dataset_stats('../data', img_size=224)
# print(f"Update config with:")
# print(f"MEAN = {mean}")
# print(f"STD = {std}")